In [0]:
from pyspark.sql.types import *
customer_schema = StructType(fields=[StructField("customer_id", IntegerType()),
                                    StructField("customer_name", StringType()),
                                    StructField("date_of_birth", DateType()),
                                    StructField("telephone", StringType()),
                                    StructField("email", StringType()),
                                    StructField("member_since", DateType()),
                                    StructField("create_timestamp", TimestampType())
])

In [0]:
df = spark.readStream.format("json") \
        .schema(customer_schema) \
        .load("/Volumes/gizmobox/landing/operational_data/customer_streaming/")
    

In [0]:
from pyspark.sql.functions import current_timestamp, col
customer_transformed_df = (
                            df.withColumn('file_path', col('_metadata.file_path'))
                              .withColumn('ingestion_date', current_timestamp())
)

In [0]:
streaming_customer = (
                        customer_transformed_df.writeStream
                                                .format('delta')
                                                .option('checkpointLocation', '/Volumes/gizmobox/landing/operational_data/customer_streaming/_checkpoint_streaming')
                                                .toTable('gizmobox.bronze.customers_stream')
)

In [0]:
%sql
select * from gizmobox.bronze.customers_stream

In [0]:
streaming_customer.stop()